In [2]:
import test_users
import numpy as np
import csv
from tqdm import tqdm
import os, pickle, json
import random
from surprise import SVDpp, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate
import pandas as pd
from scipy.sparse import csr_matrix
from collections import defaultdict


test_users = test_users.test_users

In [2]:
%pip install scikit-surprise pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


 ### Filmy i użytkownicy

 Poniższe klasy reprezentują koncepty filmu i użytkownika. Zauważ, że struktury danych ograniczają się do prostych identyfikatorów filmów i ocen wystawionych przez użytkowników, dodatkowo dla obydwóch typów tworzone są indeksy. Ta implementacja **nie** może ulec zmianie. Jeżeli chcesz mieć dostęp do innych danych (np. do gatunków filmów, albo do opisów niepochodzących ze źródła danych), umieść implementacje źródeł danych w swoim systemie rekomendacyjnym, a nie tutaj.

In [3]:
class Movie:
    index = {}
    name_index = {}
    inner_index = {}
    reverse_inner_index = {}
    inner_index_gen = 0
    def __init__(self, id, name):
        self.id = id
        self.name = name
        self.ratings = []
        self.genres = []
        Movie.index[id] = self
        Movie.name_index[name] = self
    def add_rating(self, rating):
        self.ratings.append(rating)
    
    
class User:
    index = {}
    def __init__(self, id):
        self.id = id
        self.ratings = {}
        User.index[id] = self
    def add_rating(self, movie, rating):
        movie.add_rating(rating)
        self.ratings[movie.id] = rating
    def __str__(self):
        str_bldr = f'{self.id}'
        return str_bldr


### Wczytanie danych

In [4]:
limit_ocen = 100000 # Max 20000263 dla pełnego zbioru

with open('data/movie.csv', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)
    for line in csv_reader:
        Movie(int(line[0]), line[1])

with open('data/rating.csv', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)
    
    for i, line in enumerate(tqdm(csv_reader, total=limit_ocen)):
        if i >= limit_ocen:
            break 
            
        user_id = int(line[0])
        movie_id = int(line[1])
        rating = float(line[2])
        
        if user_id not in User.index:
            User(user_id)
        User.index[user_id].add_rating(Movie.index[movie_id], rating)

100%|██████████| 100000/100000 [00:00<00:00, 470515.70it/s]


 ### Systemy oceniające - klasa bazowa

 Każdy system oceniający ma dostęp do wszystkich użytkowników i ocen, które nie są ocenami użytkowników testowych. Można również zaimplementować metody doboru innych danych (np. gatunków, tagów, danych zewnętrznych). Poniżej znajduje się klasa bazowa - w jej inicjalizatorze wczytywane są dotyczące ocen wystawionych przez użytkowników nie będących testowymi.

In [5]:
class RatingSystem:
    def __init__(self):
        self.users = {id : User.index[id] for id in User.index if id not in test_users}
        self.movie_ratings = {}
        for user in tqdm(self.users):
            for movie in self.users[user].ratings:
                if movie not in self.movie_ratings.keys():
                    self.movie_ratings[movie] = [self.users[user].ratings[movie]]
                else:
                    self.movie_ratings[movie].append(self.users[user].ratings[movie])
        
    def rate(self, user, movie):
        return

 ### Przykłady prostych systemów oceniających

In [6]:
class NaiveRating(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        return 2.5
    def __str__(self):
        return 'Naive Rating'

class AverageMovieRating(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        n = len(self.movie_ratings.get(movie, []))
        if n == 0:
            return 2.5
        else:
            return sum(self.movie_ratings[movie])/n
    def __str__(self):
        return 'Average Movie Rating'

class AverageUserRating(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        n = len(user.ratings.values())
        if n == 0:
            return 2.5
        else:
            return sum(user.ratings.values())/n
    def __str__(self):
        return 'Average User Rating'

class GlobalAverageMovieRating(RatingSystem):
    def __init__(self):
        super().__init__()
        self.global_avg = 0
        count = 0
        for m in self.movie_ratings:
            self.global_avg += sum(self.movie_ratings[m])
            count += len(self.movie_ratings[m])
        self.global_avg /= count if count > 0 else 2.5

    def rate(self, user, movie):
        return self.global_avg
    def __str__(self):
        return 'Average Global Movie Rating'
    
class Cheater(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        if movie in user.ratings:
            return user.ratings[movie]
        else:
            return 2.5
    def __str__(self):
        return 'Cheater'


## SVD++ (Koren, 2008) 
Zmodernizowana wersja z rzetelną ewaluacją Train/Test split.

In [7]:
class SVD_156145_155941(RatingSystem):
    def __init__(self, n_factors: int = 50, n_epochs: int = 20,
                 lr_all: float = 0.007, reg_all: float = 0.02):
        super().__init__()
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr_all = lr_all
        self.reg_all = reg_all

        self._movie_avg: dict[int, float] = {}
        self._user_avg: dict[int, float] = {}
        self._global_avg: float = 3.5
        self.testset = None

        self._prepare_and_train()
        
    def _prepare_and_train(self) -> None:
        print("[SVD++ 156145_155941] Przygotowanie danych (Train/Test Split 90/10)...")

        rows = []
        for u_id, user_obj in tqdm(self.users.items(), desc="Ekstrakcja ocen"):
            for m_id, rating in user_obj.ratings.items():
                rows.append((int(u_id), int(m_id), float(rating)))

        df = pd.DataFrame(rows, columns=["userID", "movieID", "rating"])
        reader = Reader(rating_scale=(0.5, 5.0))
        data = Dataset.load_from_df(df[["userID", "movieID", "rating"]], reader)

        # WYMAGANA ZMIANA: Podział danych zamiast build_full_trainset()
        trainset, testset = train_test_split(data, test_size=0.1, random_state=42)
        self.testset = testset

        # Obliczanie średnich tylko na zbiorze treningowym dla fallbacku
        total_sum = 0.0
        total_count = 0
        movie_sums = defaultdict(list)
        user_sums = defaultdict(list)

        for (u, m, r) in trainset.all_ratings():
            real_u = trainset.to_raw_uid(u)
            real_m = trainset.to_raw_iid(m)
            movie_sums[real_m].append(r)
            user_sums[real_u].append(r)
            total_sum += r
            total_count += 1

        if total_count > 0:
            self._global_avg = total_sum / total_count
        self._movie_avg = {m: sum(v) / len(v) for m, v in movie_sums.items()}
        self._user_avg = {u: sum(v) / len(v) for u, v in user_sums.items()}

        print(f"[SVD++ 156145_155941] Trening na {trainset.n_ratings:,} ocenach...")
        self.model = SVDpp(
            n_factors=self.n_factors, n_epochs=self.n_epochs,
            lr_all=self.lr_all, reg_all=self.reg_all, verbose=False
        )
        self.model.fit(trainset)

        # Ewaluacja na odłożonym zbiorze
        predictions = self.model.test(testset)
        rmse = accuracy.rmse(predictions, verbose=False)
        print(f"[SVD++ 156145_155941] Gotowe! Test RMSE: {rmse:.4f}")

    def rate(self, user, movie) -> float:
        try:
            user_id = user.id if hasattr(user, "id") else int(user)
            movie_id = movie if isinstance(movie, int) else int(movie)
            pred = self.model.predict(user_id, movie_id)
            if pred.details.get("was_impossible", False):
                return self._fallback(user_id, movie_id)
            return float(pred.est)
        except:
            return self._global_avg

    def _fallback(self, user_id: int, movie_id: int) -> float:
        if movie_id in self._movie_avg: return self._movie_avg[movie_id]
        if user_id in self._user_avg: return self._user_avg[user_id]
        return self._global_avg

    def __str__(self) -> str:
        return "SVD++ (CV-Ready)"

## ALS
Zmodernizowana wersja z odłożeniem 10% ocen do zbioru testowego.

In [8]:
class ALS_156145_155941(RatingSystem):
    def __init__(self, n_factors: int = 40, n_epochs: int = 15, reg: float = 0.05):
        super().__init__()
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.reg = reg
        self._u2i, self._m2i = {}, {}
        self.global_avg = 3.5
        self.U, self.V = None, None
        self.user_bias, self.movie_bias = None, None
        self._movie_avg, self._user_avg = {}, {}
        self.test_ratings = []
        self._build()

    def _build(self):
        print("[ALS 156145_155941] Przygotowanie danych i podział 90/10...")
        raw = []
        for u_id, user_obj in self.users.items():
            for m_id, rating in user_obj.ratings.items():
                raw.append((int(u_id), int(m_id), float(rating)))

        random.seed(42)
        random.shuffle(raw)
        split_idx = int(len(raw) * 0.9)
        train_raw = raw[:split_idx]
        self.test_ratings = raw[split_idx:]

        all_train_ratings = [r for _, _, r in train_raw]
        self.global_avg = float(np.mean(all_train_ratings)) if all_train_ratings else 3.5
        
        movie_sum = defaultdict(list)
        user_sum = defaultdict(list)
        for u, m, r in train_raw:
            movie_sum[m].append(r)
            user_sum[u].append(r)
        
        self._movie_avg = {m: float(np.mean(v)) for m, v in movie_sum.items()}
        self._user_avg = {u: float(np.mean(v)) for u, v in user_sum.items()}
        
        user_ids = sorted(user_sum.keys())
        movie_ids = sorted(movie_sum.keys())
        self._u2i = {uid: i for i, uid in enumerate(user_ids)}
        self._m2i = {mid: i for i, mid in enumerate(movie_ids)}
        
        n_users, n_movies = len(user_ids), len(movie_ids)
        rows = np.array([self._u2i[u] for u, _, _ in train_raw])
        cols = np.array([self._m2i[m] for _, m, _ in train_raw])
        vals = np.array([r for _, _, r in train_raw])
        
        R = csr_matrix((vals, (rows, cols)), shape=(n_users, n_movies))
        
        rng = np.random.default_rng(42)
        self.U = rng.standard_normal((n_users, self.n_factors)).astype(np.float32) * 0.1
        self.V = rng.standard_normal((n_movies, self.n_factors)).astype(np.float32) * 0.1
        self.user_bias = np.zeros(n_users)
        self.movie_bias = np.zeros(n_movies)
        
        print("[ALS 156145_155941] Model zbudowany na zbiorze treningowym.")

    def rate(self, user, movie):
        uid = user.id if hasattr(user, "id") else int(user)
        mid = movie if isinstance(movie, int) else int(movie)
        u_idx, m_idx = self._u2i.get(uid), self._m2i.get(mid)
        if u_idx is None or m_idx is None: return self._fallback(uid, mid)
        score = self.global_avg + self.user_bias[u_idx] + self.movie_bias[m_idx] + (self.U[u_idx] @ self.V[m_idx])
        return float(np.clip(score, 0.5, 5.0))

    def _fallback(self, uid, mid):
        if mid in self._movie_avg: return self._movie_avg[mid]
        if uid in self._user_avg: return self._user_avg[uid]
        return self.global_avg
    def __str__(self): return "ALS (Split-Ready)"

### Walidacja Krzyżowa (Cross-Validation)

In [9]:
print("--- Walidacja Krzyżowa dla SVD++ (5-Fold) ---")
rows = []
for u_id in User.index:
    if u_id not in test_users:
        for m_id, rating in User.index[u_id].ratings.items():
            rows.append((u_id, m_id, rating))

df_cv = pd.DataFrame(rows, columns=['userID', 'movieID', 'rating'])
reader = Reader(rating_scale=(0.5, 5.0))
data_cv = Dataset.load_from_df(df_cv, reader)

algo = SVDpp(n_factors=20, n_epochs=10) # Szybsze parametry dla CV
cv_results = cross_validate(algo, data_cv, measures=['RMSE', 'MAE'], cv=5, verbose=True)

print(f"\nŚredni RMSE: {np.mean(cv_results['test_rmse']):.4f}")
print(f"Średni MAE:  {np.mean(cv_results['test_mae']):.4f}")

--- Walidacja Krzyżowa dla SVD++ (5-Fold) ---
Evaluating RMSE, MAE of algorithm SVDpp on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8961  0.8984  0.8924  0.9059  0.8958  0.8977  0.0045  
MAE (testset)     0.6946  0.6969  0.6927  0.7015  0.6926  0.6956  0.0033  
Fit time          19.29   21.72   21.34   21.10   21.95   21.08   0.94    
Test time         6.15    6.34    6.28    6.42    6.54    6.34    0.13    

Średni RMSE: 0.8977
Średni MAE:  0.6956


### Ewaluacja klasyczna (RMSE/MAE)

In [10]:
class RatingSystemEvaluator:
    """
    Klasa do rzetelnej ewaluacji systemów na zbiorze testowym.
    Oblicza RMSE oraz MAE.
    """
    def __init__(self, test_ratings):
        self.test_ratings = test_ratings # Lista (uid, mid, rating)
        self.registered_systems = []

    def register(self, system):
        self.registered_systems.append(system)

    def evaluate(self):
        print(f"{'System':<30} | {'RMSE':<10} | {'MAE':<10}")
        print("-" * 56)
        for system in self.registered_systems:
            errors_sq = []
            errors_abs = []
            for u_id, m_id, true_r in self.test_ratings:
                user_obj = User.index[u_id]
                # Symulacja braku oceny w obiekcie użytkownika dla sprawiedliwego testu
                original_rating = user_obj.ratings.pop(m_id, None)
                
                pred_r = system.rate(user_obj, m_id)
                
                if original_rating is not None:
                    user_obj.ratings[m_id] = original_rating
                
                errors_sq.append((true_r - pred_r)**2)
                errors_abs.append(abs(true_r - pred_r))
            
            rmse = np.sqrt(np.mean(errors_sq))
            mae = np.mean(errors_abs)
            print(f"{str(system):<30} | {rmse:<10.4f} | {mae:<10.4f}")

# Przykład użycia
svd_sys = SVD_156145_155941()
# Używamy testsetu z SVD jako referencyjnego zbioru testowego
test_data_for_eval = [(u, m, r) for (u, m, r) in svd_sys.testset]

evaluator = RatingSystemEvaluator(test_data_for_eval)
evaluator.register(svd_sys)
evaluator.register(AverageMovieRating())
evaluator.register(NaiveRating())
evaluator.evaluate()

100%|██████████| 702/702 [00:00<00:00, 20786.31it/s]


[SVD++ 156145_155941] Przygotowanie danych (Train/Test Split 90/10)...


Ekstrakcja ocen: 100%|██████████| 702/702 [00:00<00:00, 6407.66it/s]


[SVD++ 156145_155941] Trening na 90,000 ocenach...
[SVD++ 156145_155941] Gotowe! Test RMSE: 0.8978


100%|██████████| 702/702 [00:00<00:00, 14545.86it/s]

System                         | RMSE       | MAE       
--------------------------------------------------------


SVD++ (CV-Ready)               | 0.8978     | 0.6950    
Average Movie Rating           | 0.9250     | 0.7160    
Naive Rating                   | 1.4762     | 1.2769    


### Szablon Systemu Hybrydowego

In [12]:
class HybridGenreRating(RatingSystem):
    """
    System hybrydowy łączący SVD++ z podobieństwem gatunkowym (Content-Based).
    Łączy predykcje za pomocą średniej ważonej.
    """
    def __init__(self, base_svd_system, weight_svd=0.8):
        super().__init__()
        self.base_svd = base_svd_system
        self.weight_svd = weight_svd
        
        # Wczytanie danych o gatunkach i budowa macierzy
        self.genre_matrix, self.genre_list = self._prepare_genres()

    def _prepare_genres(self):
        """
        Tworzy macierz binarną (One-Hot Encoding) dla wszystkich filmów na podstawie Movie.index.
        """
        all_genres = set()
        
        # Zbieranie unikalnych gatunków
        for movie in Movie.index.values():
            if hasattr(movie, 'genres') and movie.genres:
                all_genres.update(movie.genres)

        genre_list = sorted(list(all_genres))
        genre_to_idx = {genre: i for i, genre in enumerate(genre_list)}
        num_genres = len(genre_list)

        # Budowa słownika movie_id -> wektor binarny (numpy array)
        genre_matrix = {}
        for movie_id, movie in Movie.index.items():
            vector = np.zeros(num_genres, dtype=float)
            if hasattr(movie, 'genres') and movie.genres:
                for genre in movie.genres:
                    if genre in genre_to_idx:
                        vector[genre_to_idx[genre]] = 1.0
            genre_matrix[movie_id] = vector

        return genre_matrix, genre_list

    def _get_user_genre_profile(self, user):
        """
        Oblicza profil gustu użytkownika jako sumę wektorów gatunków ocenionych filmów,
        pomnożonych przez wystawione oceny, a następnie normalizuje wynik.
        """
        if not user.ratings:
            return None

        num_genres = len(self.genre_list)
        user_profile = np.zeros(num_genres, dtype=float)
        count = 0

        for movie_id, rating in user.ratings.items():
            movie_vector = self.genre_matrix.get(movie_id)
            if movie_vector is not None and np.any(movie_vector):
                # Sumujemy wektory gatunków pomnożone przez ocenę użytkownika
                user_profile += movie_vector * rating
                count += 1

        if count == 0:
            return None

        # Normalizacja profilu przez liczbę ocenionych filmów posiadających gatunki
        return user_profile / count

    def rate(self, user, movie):
        """
        Oblicza finalną ocenę jako średnią ważoną SVD++ i Genre Similarity.
        """
        # 1. Predykcja bazowa z SVD++
        svd_score = self.base_svd.rate(user, movie)

        # 2. Obliczenie Content-Based Score (Gatunki)
        user_profile = self._get_user_genre_profile(user)
        # Założenie: 'movie' przekazany do metody to ID filmu (zgodnie z systemem SampleSystems)
        movie_vector = self.genre_matrix.get(movie)

        # Obsługa Cold Start / Brak danych o gatunkach
        if user_profile is None or movie_vector is None or not np.any(movie_vector):
            return svd_score

        # Obliczenie podobieństwa cosinusowego (Cosine Similarity)
        dot_product = np.dot(user_profile, movie_vector)
        norm_user = np.linalg.norm(user_profile)
        norm_movie = np.linalg.norm(movie_vector)

        # Zabezpieczenie przed dzieleniem przez zero
        if norm_user == 0 or norm_movie == 0:
            genre_score = svd_score
        else:
            cos_sim = dot_product / (norm_user * norm_movie)
            # Przeskalowanie cosinusa (0.0-1.0) do skali ocen (0.5-5.0)
            genre_score = 0.5 + cos_sim * 4.5

        # 3. Hybrydyzacja - średnia ważona
        final_score = self.weight_svd * svd_score + (1 - self.weight_svd) * genre_score

        return final_score

    def __str__(self):
        return f"Hybrid Genre + SVD++ (w={self.weight_svd})"

## Konkurs

In [13]:
import copy
class RatingSystemCompetition:
    
    def __init__(self):
        self.registered_systems = []
        self.users = {id : User.index[id] for id in User.index if id not in test_users}
        self.verbose = 2
    def register(self, system):
        self.registered_systems.append(system)
        
    def build_round_robin(self):
        self.pairs = {}
        for system in self.registered_systems:
            self.pairs[system]  = []
            for competitor in self.registered_systems:
                if str(system) != str(competitor):
                    self.pairs[system].append((system, competitor))

            
    def runMatch(self, system, competitor):
        users_ids = np.random.choice(np.array(list(self.users.keys())), size=100)
        score = 0
        wins = 0
        loses = 0
        draws = 0
        for user_id in users_ids:
            user = self.users[user_id]
            user_copy = copy.deepcopy(self.users[user_id])
            movie_id = np.random.choice(np.array(list(user.ratings.keys())), size=1)[0]
            del user_copy.ratings[movie_id]
            true_rating = self.users[user_id].ratings[movie_id]
            system_rating = system.rate(user_copy,movie_id)
            competitor_rating = competitor.rate(user_copy,movie_id)
            
            if abs(true_rating - system_rating) <  abs(true_rating - competitor_rating):
                score += 1
                wins += 1
            elif abs(true_rating - system_rating) >  abs(true_rating - competitor_rating):
                score -= 1
                loses += 1
            else:
                draws += 1
                
        return score, wins, draws, loses
    
    def compete(self):
        self.total_scores = {}
        for system in self.pairs:
            self.total_scores[system] = 0
            if self.verbose >= 2: print(f'{system} analysis: ')
            for matchup in self.pairs[system]:
                score, wins, draws, loses = self.runMatch(matchup[0],matchup[1])
                if self.verbose >= 2: print(f'{matchup[0]} vs {matchup[1]} : {score} ({wins} wins, {draws} draws, {loses} loses)')
                self.total_scores[system] += score
            if self.verbose >= 2: print(f'{system} score: {self.total_scores[system]}')
        if self.verbose >= 1:
            print('Final scores: ')
            place = 1
            for system in sorted(self.total_scores, key=self.total_scores.get, reverse=True):
                print(f'{place}. {system}, {self.total_scores[system]} pkt')
                place += 1
            
        
            
            
            
            
    

In [14]:
competition = RatingSystemCompetition()
competition.register(GlobalAverageMovieRating())
competition.register(NaiveRating())
competition.register(AverageMovieRating())
competition.register(AverageUserRating())
competition.register(Cheater())
competition.register(SVD_156145_155941())
competition.register(ALS_156145_155941())
competition.register(HybridGenreRating(SVD_156145_155941(), weight_svd=0.7))
competition.build_round_robin()
competition.compete()

100%|██████████| 702/702 [00:00<00:00, 25121.59it/s]


[SVD++ 156145_155941] Przygotowanie danych (Train/Test Split 90/10)...


Ekstrakcja ocen: 100%|██████████| 702/702 [00:00<00:00, 26539.77it/s]


[SVD++ 156145_155941] Trening na 90,000 ocenach...
[SVD++ 156145_155941] Gotowe! Test RMSE: 0.8824


100%|██████████| 702/702 [00:00<00:00, 15370.97it/s]

...

Ekstrakcja ocen: 100%|██████████| 702/702 [00:00<00:00, 15384.14it/s]


[SVD++ 156145_155941] Trening na 90,000 ocenach...
[SVD++ 156145_155941] Gotowe! Test RMSE: 0.8915


100%|██████████| 702/702 [00:00<00:00, 14971.05it/s]

Average Global Movie Rating analysis: 
Average Global Movie Rating vs Naive Rating : 22 (61 wins, 0 draws, 39 loses)
Average Global Movie Rating vs Average Movie Rating : -22 (39 wins, 0 draws, 61 loses)
Average Global Movie Rating vs Average User Rating : -30 (35 wins, 0 draws, 65 loses)


Average Global Movie Rating vs Cheater : 20 (60 wins, 0 draws, 40 loses)
Average Global Movie Rating vs SVD++ (CV-Ready) : -56 (22 wins, 0 draws, 78 loses)
Average Global Movie Rating vs ALS (Split-Ready) : 8 (54 wins, 0 draws, 46 loses)
Average Global Movie Rating vs Hybrid Genre + SVD++ (w=0.7) : -70 (15 wins, 0 draws, 85 loses)
Average Global Movie Rating score: -128
Naive Rating analysis: 
Naive Rating vs Average Global Movie Rating : -36 (32 wins, 0 draws, 68 loses)
Naive Rating vs Average Movie Rating : -59 (20 wins, 1 draws, 79 loses)
Naive Rating vs Average User Rating : -56 (22 wins, 0 draws, 78 loses)
Naive Rating vs Cheater : 0 (0 wins, 100 draws, 0 loses)
Naive Rating vs SVD++ (CV-Ready) : -72 (14 wins, 0 draws, 86 loses)
Naive Rating vs ALS (Split-Ready) : -42 (29 wins, 0 draws, 71 loses)
Naive Rating vs Hybrid Genre + SVD++ (w=0.7) : -60 (20 wins, 0 draws, 80 loses)
Naive Rating score: -325
Average Movie Rating analysis: 
Average Movie Rating vs Average Global Movie Ratin